# Clase 4 · Parte 2: Primer contacto con IA para imágenes

En la parte 1 aprendimos el patrón que vamos a usar todo el curso: recorrer datos con un `for`, guardar una ficha en cada vuelta y armar una tabla al final.

Hoy cambiamos dos cosas:
- en lugar de testimonios, vamos a analizar **fotos**;
- en lugar de contar palabras, le vamos a pedir a un **modelo de IA** que diga qué ve.

No vamos a entrenar nada. Vamos a usar un modelo que otras personas ya entrenaron (un **modelo preentrenado**) y vamos a mirar con cuidado qué responde.

## 1. Una foto es una tabla de números

Antes de usar IA, veamos qué es una foto para la computadora. Primero traemos tres librerías:

- `requests` descarga cosas de internet.
- `PIL` (Pillow) abre y muestra imágenes.
- `numpy` trabaja con tablas de números.

In [2]:
import requests
from PIL import Image
import numpy as np
import pandas as pd

In [ ]:
enlace = "http://images.cocodataset.org/val2017/000000039769.jpg"

foto = Image.open(requests.get(enlace, stream=True).raw)
foto

In [4]:
pixeles = np.array(foto)
print(pixeles.shape)

(480, 640, 3)


Igual que con `tabla.shape`, aquí `shape` nos dice el tamaño. Vas a ver tres números:

| Número | Qué es |
|---|---|
| el primero | filas: el alto de la foto, en píxeles |
| el segundo | columnas: el ancho de la foto, en píxeles |
| el tercero | 3 canales de color: rojo, verde y azul |

Cada píxel guarda tres números entre 0 y 255: cuánto rojo, cuánto verde y cuánto azul tiene. Veamos el píxel de la esquina superior izquierda:

In [5]:
pixeles[0, 0]

array([140,  25,  56], dtype=uint8)

`pixeles[0, 0]` se lee: fila 0, columna 0. Igual que `ciudades[0]` en una lista, en Python se empieza a contar desde 0.

También podemos quedarnos con un pedazo de la foto, como cuando filtramos filas de la tabla. `0:200` significa "desde la fila 0 hasta antes de la 200":

In [ ]:
recorte = pixeles[0:200, 0:300]
Image.fromarray(recorte)

### Tu turno
1. Imprime el píxel de la fila 200, columna 300. ¿Qué color crees que es?
2. Haz un recorte distinto de la foto y muéstralo con `Image.fromarray(...)`.

**Para pensar:** si para la computadora una foto es solo una tabla de números, ¿cómo podría "saber" que ahí hay gatos?

In [ ]:
# escribe aquí

In [ ]:
# escribe aquí

## 2. Pedirle a un modelo que describa una foto

Un **modelo de clasificación de imágenes** recibe esa tabla de números y devuelve **etiquetas**: palabras que, según lo que aprendió, describen la foto.

Vamos a usar la librería `transformers`, que ya viene en Colab. Su herramienta `pipeline` prepara un modelo listo para usar:

In [7]:
from transformers import pipeline

clasificador = pipeline("image-classification", model="google/vit-base-patch16-224")

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Se lee así:
- `pipeline` es la herramienta que arma el modelo.
- `"image-classification"` es la tarea: clasificar imágenes.
- `model="google/vit-base-patch16-224"` es el modelo que elegimos. Lo entrenó Google con una base de fotos llamada **ImageNet**.

La primera vez tarda un poco porque Colab descarga el modelo. Ahora le pasamos la foto:

In [8]:
respuesta = clasificador(enlace)
respuesta

[{'label': 'Egyptian cat', 'score': 0.9374894499778748},
 {'label': 'tabby, tabby cat', 'score': 0.038421303033828735},
 {'label': 'tiger cat', 'score': 0.014402329921722412},
 {'label': 'lynx, catamount', 'score': 0.003274730173870921},
 {'label': 'Siamese cat, Siamese', 'score': 0.0006791056366637349}]

Fíjate en la forma de la respuesta: es una **lista de fichas**, justo lo que vimos en la parte 1. Cada ficha tiene dos campos:

- `"label"`: la etiqueta, lo que el modelo cree que ve.
- `"score"`: un número entre 0 y 1 que indica qué tan **seguro** está el modelo de esa etiqueta.

Ojo: `score` no es la probabilidad de que el modelo tenga razón. Un modelo puede estar muy seguro y equivocarse.

Como es una lista de fichas, ya sabemos sacar partes:

In [9]:
primera = respuesta[0]
print(primera["label"])
print(primera["score"])

Egyptian cat
0.9374894499778748


### Tu turno
1. Imprime la etiqueta de la **segunda** ficha de `respuesta`.
2. Convierte `respuesta` en una tabla con `pd.DataFrame(respuesta)`.

In [ ]:
# escribe aquí

In [ ]:
# escribe aquí

## 3. El mismo patrón de siempre, ahora con fotos

Ahora tenemos varias fotos. Recuerda el patrón de la parte 1: lista vacía, `for`, ficha, `append`, tabla. Lo único nuevo es la línea donde llamamos al modelo.

Si algún enlace no carga, reemplázalo por otro enlace a una imagen `.jpg` o `.png`.

In [10]:
fotos = [
    "http://images.cocodataset.org/val2017/000000039769.jpg",
    "http://images.cocodataset.org/val2017/000000000285.jpg",
    "http://images.cocodataset.org/val2017/000000000724.jpg",
    "https://huggingface.co/datasets/Narsil/image_dummy/raw/main/parrots.png",
]

In [11]:
resultados = []

for enlace in fotos:
    respuesta = clasificador(enlace)
    mejor = respuesta[0]
    ficha = {"foto": enlace, "etiqueta": mejor["label"], "confianza": round(mejor["score"], 2)}
    resultados.append(ficha)

clasificacion = pd.DataFrame(resultados)
clasificacion

,foto,etiqueta,confianza
0,http://images.cocodataset.org/val2017/00000003...,Egyptian cat,0.94
1,http://images.cocodataset.org/val2017/00000000...,"brown bear, bruin, Ursus arctos",1.00
2,http://images.cocodataset.org/val2017/00000000...,street sign,0.99
3,https://huggingface.co/datasets/Narsil/image_d...,macaw,0.99


Compara con lo que hicimos con los testimonios:

| | Parte 1 · Testimonios | Parte 2 · Fotos |
|---|---|---|
| Lo que recorro | `tabla["testimonio"]` | `fotos` |
| Lo que hago en cada vuelta | `len(testimonio.split())` | `clasificador(enlace)` |
| Lo que guardo | una ficha | una ficha |
| Al final | `pd.DataFrame(resultados)` | `pd.DataFrame(resultados)` |

La estructura es la misma. Eso es lo que vamos a repetir con documentos y con audios.

### Tu turno
1. Sube a Colab una foto tuya o de tu trabajo de campo (una calle, un mercado, una fiesta, un objeto). Usa el ícono de carpeta, como hicimos con `entrevistas.csv`.
2. Pásala por el clasificador. En lugar del enlace, escribe el nombre del archivo, por ejemplo `clasificador("mercado.jpg")`.
3. Agrega el nombre de tu archivo a la lista `fotos` y vuelve a ejecutar el `for`.

In [ ]:
# escribe aquí

In [ ]:
# escribe aquí

## 4. ¿Qué puede ver este modelo y qué no?

Este modelo solo conoce **1000 etiquetas**, todas en inglés, que vienen de ImageNet. Puedes ver algunas:

In [12]:
etiquetas = list(clasificador.model.config.id2label.values())
print(len(etiquetas))
print(etiquetas[:30])

1000
['tench, Tinca tinca', 'goldfish, Carassius auratus', 'great white shark, white shark, man-eater, man-eating shark, Carcharodon carcharias', 'tiger shark, Galeocerdo cuvieri', 'hammerhead, hammerhead shark', 'electric ray, crampfish, numbfish, torpedo', 'stingray', 'cock', 'hen', 'ostrich, Struthio camelus', 'brambling, Fringilla montifringilla', 'goldfinch, Carduelis carduelis', 'house finch, linnet, Carpodacus mexicanus', 'junco, snowbird', 'indigo bunting, indigo finch, indigo bird, Passerina cyanea', 'robin, American robin, Turdus migratorius', 'bulbul', 'jay', 'magpie', 'chickadee', 'water ouzel, dipper', 'kite', 'bald eagle, American eagle, Haliaeetus leucocephalus', 'vulture', 'great grey owl, great gray owl, Strix nebulosa', 'European fire salamander, Salamandra salamandra', 'common newt, Triturus vulgaris', 'eft', 'spotted salamander, Ambystoma maculatum', 'axolotl, mud puppy, Ambystoma mexicanum']
